In [1]:
# ============================================================
# CS781 Capstone - PHASE 4: Court Judgment Prediction (CJPE) - binary 0/1, Macro-F1
# Long judgments -> chunks of MAX_LEN tokens -> one vector per chunk -> learned document pooling -> 0/1
# Starts from YOUR Phase-1 model. Everything from scratch in plain PyTorch (no peft) - Phase 3+ rule.
#
# One notebook for all experiments - change Cell 1 only:
#   FT_METHOD: "full" | "partial" | "unfreeze" | "lora" | "lorma" | "adalora" | "adapter"
#   Knowledge distillation: set KD_TEACHER to a trained Phase-4 run's best_cjpe.pt
# ============================================================

# %% 0. IMPORTS
import os, json, math, time, glob, random, zipfile
from collections import Counter
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PreTrainedTokenizerFast   # loads YOUR tokenizer file only

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# %% 1. CONFIG  <-------------------------------------------------------- EDIT HERE
QUICK_TEST = False          # True = few docs, 1 epoch, to check the pipeline

MODEL_TYPE = "gpt"        # "gpt" (Model 1) | "llama" (Model 2) | "gqa" (Model 3) | "hybrid" (Model 4)
PHASE1_CKPT = "/kaggle/input/datasets/kavyajain123987/phase1-models/best bpe.pt"
TOKENIZER = "/kaggle/input/datasets/kavyajain123987/phase1/tokenizer (1).json"

FT_METHOD = "partial"          # full | partial | unfreeze | lora | lorma | adalora | adapter
FREEZE_BOTTOM = 4           # partial: freeze embeddings + this many lowest blocks
LORA_R, LORA_ALPHA = 8, 16  # lora / lorma rank + scaling
ADA_R_INIT, ADA_R_TARGET = 12, 4
ADAPTER_DIM = 64

# ---- long-document representation ----
SECTION = "tail"            # which part of the judgment is read: "tail" (last part) | "head" | "headtail" | "spread"
MAX_LEN = 512               # tokens per chunk
N_CHUNKS = 6                # chunks per document (6 x 512 = ~3k tokens per case)
CHUNK_POOL = "last"         # chunk vector: "last" token (causal model has read the whole chunk) | "mean"
DOC_POOL = "attn"           # chunks -> document: "attn" (learned attention) | "mean" | "gru"

# ---- training ----
MAX_TRAIN_DOCS = 8000       # the CJPE train set is large; a random subset keeps one run ~1 hour on a T4
EPOCHS = 2
LR_BODY = 5e-5              # full / partial / unfreeze body
LR_NEW = 1e-3               # head + adapter parameters
WEIGHT_DECAY = 0.01
DOCS_PER_STEP = 8           # gradient accumulation
WIN_BATCH = 6               # chunks per forward pass
CLASS_WEIGHT = True         # balance the two classes in the loss (Macro-F1 cares about both)
TUNE_THRESHOLD = True       # choose the decision threshold that maximises dev Macro-F1 (calibration)
EVAL_DEV_DOCS = 1200        # evaluations DURING training use a fixed random dev subset (full dev = ~6 min each);
                            # the best checkpoint is re-scored on the FULL dev set at the end to set the threshold
SEED = 42

# ---- knowledge distillation (mandatory experiment) ----
KD_TEACHER = None           # path to a trained Phase-4 run's best_cjpe.pt (file, .zip, or extracted folder)
KD_KEEP_EVERY = 2           # student = Phase-1 model keeping every 2nd block
KD_T, KD_ALPHA = 2.0, 0.5

OUTPUT_DIR = f"/kaggle/working/phase4_{MODEL_TYPE}_{FT_METHOD}" + ("_kd" if KD_TEACHER else "")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# %% 2. DOWNLOAD PHASE-4 DATA + LOOK AT IT
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["Phase_4_CJPE/*"], local_dir="/tmp/cs781_data")
P4 = os.path.join(DATA_DIR, "Phase_4_CJPE")
for root, _, files in os.walk(P4):
    for f in sorted(files):
        print(os.path.join(root, f), f"{os.path.getsize(os.path.join(root, f)) / 1e6:.1f} MB")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def peek(obj, n=300):
    s = json.dumps(obj, ensure_ascii=False)
    return s[:n] + ("..." if len(s) > n else "")

def parse(obj):
    """-> list of {"id", "text", "label" (int or None)}"""
    if isinstance(obj, dict) and "data" in obj and isinstance(obj["data"], (list, dict)):
        obj = obj["data"]
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    out = []
    for k, d in items:
        if isinstance(d, str):
            d = {"text": d}
        if not isinstance(d, dict):
            continue
        text = next((d[x] for x in ("text", "document", "case_text", "content") if x in d), None)
        if isinstance(text, list):
            text = " ".join(map(str, text))
        lab = next((d[x] for x in ("label", "labels", "pred", "outcome", "y") if x in d), None)
        out.append({"id": str(d.get("id", d.get("case_id", d.get("doc_id", k)))), "text": text or "",
                    "label": int(lab) if lab is not None else None})
    return out

splits = {}
for name, path in [("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")]:
    raw = load_json(os.path.join(P4, path))
    print(f"\n{path}: {type(raw).__name__} with {len(raw)} entries | first: "
          f"{peek(raw[0] if isinstance(raw, list) else next(iter(raw.items())))}")
    splits[name] = parse(raw)
for name, ds in splits.items():
    chars = [len(d["text"]) for d in ds]
    print(f"{name}: {len(ds)} cases | labels {dict(Counter(d['label'] for d in ds))} | "
          f"chars median {int(np.median(chars))}, max {max(chars)}")
assert splits["test"] and all(d["text"] for d in splits["test"]), "test cases not parsed - check the format above"
assert all(d["label"] in (0, 1) for d in splits["train"] + splits["dev"]), "labels must be 0/1"

random.Random(SEED).shuffle(splits["train"])
if MAX_TRAIN_DOCS:
    splits["train"] = splits["train"][:MAX_TRAIN_DOCS]
if QUICK_TEST:
    splits["train"], splits["dev"] = splits["train"][:40], splits["dev"][:20]
    EPOCHS = 1
print(f"using {len(splits['train'])} training cases, {len(splits['dev'])} dev cases")

# %% 3. TOKENIZE ONLY THE PART OF EACH JUDGMENT THAT IS READ
tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER)
tokenizer.model_max_length = int(1e9)
BUDGET = N_CHUNKS * MAX_LEN
CHAR_BUDGET = BUDGET * 8          # generous characters-per-token margin, so we never tokenize whole 100k-char cases

def select_ids(text):
    """-> list of N_CHUNKS (or fewer) token-id chunks of <= MAX_LEN tokens"""
    if SECTION == "tail":
        ids = tokenizer.encode(text[-CHAR_BUDGET:], add_special_tokens=False)[-BUDGET:]
    elif SECTION == "head":
        ids = tokenizer.encode(text[:CHAR_BUDGET], add_special_tokens=False)[:BUDGET]
    elif SECTION == "headtail":
        h = tokenizer.encode(text[:CHAR_BUDGET // 2], add_special_tokens=False)[:BUDGET // 2]
        t = tokenizer.encode(text[-CHAR_BUDGET // 2:], add_special_tokens=False)[-(BUDGET - len(h)):]
        ids = h + t if len(text) > CHAR_BUDGET else tokenizer.encode(text, add_special_tokens=False)[-BUDGET:]
    elif SECTION == "spread":   # N_CHUNKS windows evenly spread over the WHOLE judgment
        ids_all = tokenizer.encode(text, add_special_tokens=False)
        if len(ids_all) <= BUDGET:
            ids = ids_all
        else:
            starts = np.linspace(0, len(ids_all) - MAX_LEN, N_CHUNKS).astype(int)
            return [ids_all[s:s + MAX_LEN] for s in starts]
    else:
        raise ValueError(SECTION)
    ids = ids or [0]
    return [ids[i:i + MAX_LEN] for i in range(0, len(ids), MAX_LEN)]

t0 = time.time()
for name, ds in splits.items():
    for d in ds:
        d["chunks"] = select_ids(d["text"])
        d["text"] = None              # free memory
    print(f"{name}: tokenized in {time.time() - t0:.0f}s | chunks per case: "
          f"mean {np.mean([len(d['chunks']) for d in ds]):.1f}")

# %% 4. PHASE-1 MODEL DEFINITIONS (unchanged from Phase 1)
def build_model1(cfg):
    # ---------- Model 1: GPT-style ----------
    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            assert d_model % n_head == 0
            self.d_model = d_model
            self.n_head = n_head
            self.head_dim = d_model // n_head
            self.c_attn = nn.Linear(d_model, 3 * d_model)
            self.c_proj = nn.Linear(d_model, d_model)
            self.attn_dropout = dropout
            self.resid_dropout = nn.Dropout(dropout)

        def forward(self, x):
            B, T, C = x.shape
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.d_model, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=(self.attn_dropout if self.training else 0.0),
                is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.c_proj(y)
            y = self.resid_dropout(y)
            return y


    class MLP(nn.Module):
        def __init__(self, d_model, dropout):
            super().__init__()
            self.fc = nn.Linear(d_model, 4 * d_model)
            self.proj = nn.Linear(4 * d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            x = self.fc(x)
            x = F.gelu(x)
            x = self.proj(x)
            x = self.dropout(x)
            return x


    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            self.ln_1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, dropout)
            self.ln_2 = nn.LayerNorm(d_model)
            self.mlp = MLP(d_model, dropout)

        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x


    class LegalGPT(nn.Module):
        def __init__(self, vocab_size, block_size, n_layer, n_head, d_model, dropout):
            super().__init__()
            self.block_size = block_size
            self.wte = nn.Embedding(vocab_size, d_model)
            self.wpe = nn.Embedding(block_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.wte.weight
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            assert T <= self.block_size
            positions = torch.arange(0, T, device=idx.device)
            x = self.wte(idx) + self.wpe(positions)
            for block in self.blocks:
                x = block(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return LegalGPT(**cfg)


def build_model2(cfg):
    # ---------- Model 2: LLaMA-style ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = CausalSelfAttention(d_model, n_heads, max_seq_len)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([LLaMABlock(cfg.d_model, cfg.n_heads, cfg.ffn_dim, cfg.max_seq_len)
                                         for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
        # (your generate() method is not needed: the pipeline's batched generate_batch() is used)
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model3(cfg):
    # ---------- Model 3: LLaMA + GQA + sliding window ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class GQASlidingWindowAttention(nn.Module):
        """
        GQA: n_heads query heads share n_kv_heads key/value heads (8 Q -> 2 KV: each KV head serves 4 Q heads).
        Sliding window: token i attends only to tokens j with 0 <= i - j < window_size (causal + local).
        """
        def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, window_size):
            super().__init__()
            assert d_model % n_heads == 0
            assert n_heads % n_kv_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.n_kv_heads = n_kv_heads
            self.head_dim = d_model // n_heads
            self.window_size = window_size
            self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
            # mask[q, k] = True -> q may attend to k   (k <= q  AND  q - k < window)
            positions = torch.arange(max_seq_len)
            distance = positions[:, None] - positions[None, :]
            self.register_buffer("window_mask", (distance >= 0) & (distance < window_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # [B, 8, T, D]
            k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            q, k = self.rope(q, k)
            repeat_factor = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(repeat_factor, dim=1)                                  # [B, 8, T, D]
            v = v.repeat_interleave(repeat_factor, dim=1)
            mask = self.window_mask[:T, :T]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, max_seq_len, window_size):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = GQASlidingWindowAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads,
                                                       max_seq_len=max_seq_len, window_size=window_size)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([
                LLaMABlock(d_model=cfg.d_model, n_heads=cfg.n_heads, n_kv_heads=cfg.n_kv_heads, ffn_dim=cfg.ffn_dim,
                           max_seq_len=cfg.max_seq_len, window_size=cfg.window_size)
                for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model4(cfg):
    # ---------- Model 4: hybrid attention + gated conv ----------
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles."""
        def __init__(self, dim, max_seq_len=2048, base=10000):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            self.base = base
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.shape[-2]
            if seq_len > self.cos_cached.shape[-2]:
                self._build_cache(seq_len)
            # FIX: cast to q's dtype. Otherwise fp16 q * fp32 cos -> fp32 q/k while v stays fp16,
            # and scaled_dot_product_attention can fail with a dtype mismatch.
            cos = self.cos_cached[:, :, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :, :seq_len, :].to(q.dtype)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            return q, k


    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return x * rms * self.weight


    class SwiGLU(nn.Module):
        def __init__(self, d_model, hidden_dim):
            super().__init__()
            self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
            self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
            self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

        def forward(self, x):
            return self.w2(F.silu(self.w1(x)) * self.w3(x))


    class CausalSelfAttention(nn.Module):
        """GLOBAL mixer: token i attends to tokens 0..i."""
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len=max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class GatedCausalConv(nn.Module):
        """
        LOCAL mixer: Linear(D->2D) -> (value, gate);
        value -> causal depthwise Conv1d (left padding only: position t sees t-k+1 .. t, never t+1);
        out = Linear( conv(value) * sigmoid(gate) ).
        """
        def __init__(self, d_model, kernel_size=5):
            super().__init__()
            self.d_model = d_model
            self.kernel_size = kernel_size
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)

        def forward(self, x):
            value, gate = self.in_proj(x).chunk(2, dim=-1)
            value = value.transpose(1, 2)                          # [B, D, T] for Conv1d
            value = F.pad(value, (self.kernel_size - 1, 0))        # LEFT padding only -> causal
            value = self.conv(value).transpose(1, 2)               # back to [B, T, D]
            return self.out_proj(value * torch.sigmoid(gate))


    class HybridBlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len, conv_kernel, use_attention):
            super().__init__()
            self.use_attention = use_attention
            self.norm1 = RMSNorm(d_model)
            self.mixer = (CausalSelfAttention(d_model=d_model, n_heads=n_heads, max_seq_len=max_seq_len)
                          if use_attention else GatedCausalConv(d_model=d_model, kernel_size=conv_kernel))
            self.norm2 = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model=d_model, hidden_dim=ffn_dim)

        def forward(self, x):
            x = x + self.mixer(self.norm1(x))
            x = x + self.ffn(self.norm2(x))
            return x


    class HybridLM(nn.Module):
        def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, max_seq_len, conv_kernel):
            super().__init__()
            self.vocab_size = vocab_size
            self.d_model = d_model
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            # even layers -> attention, odd layers -> gated conv
            self.blocks = nn.ModuleList([
                HybridBlock(d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim, max_seq_len=max_seq_len,
                            conv_kernel=conv_kernel, use_attention=(i % 2 == 0))
                for i in range(n_layers)])
            self.final_norm = RMSNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, input_ids, targets=None):
            x = self.token_embedding(input_ids)
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                # FIX: the pipeline's targets are ALREADY shifted by one (y = x[1:]).
                # Your version shifted again (logits[:, :-1] vs targets[:, 1:]) -> the model
                # would learn to predict 2 tokens ahead. Same loss as Models 1-3 now.
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return HybridLM(**cfg)


BUILDERS = {"gpt": build_model1, "llama": build_model2, "gqa": build_model3, "hybrid": build_model4}

# %% 5. ADAPTATION METHODS (from scratch)
class LoRALinear(nn.Module):
    """LoRA:   y = W x + s * B A x,  s = alpha/r.   rsLoRA: same but s = alpha/sqrt(r) (rank-stabilised)."""
    def __init__(self, base, r, alpha, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative):  y = (I + s * B A) W x.  Check the exact variant against the LoRMA paper."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA:  y = W x + s * P diag(lambda * mask) Q x  (SVD-form update).
    Ranks are pruned during training by importance |lambda * dL/dlambda| (EMA), down to a global budget."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("mask", torch.ones(r))
        self.register_buffer("importance", torch.zeros(r))

    def forward(self, x):
        h = F.linear(x, self.Q) * (self.lam * self.mask)
        return self.base(x) + self.scale * F.linear(h, self.P)

    def orth_reg(self):
        r = self.lam.numel()
        I = torch.eye(r, device=self.P.device)
        return ((self.P.T @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.T - I) ** 2).sum()


def wrap_linears(module, fn):
    """replace every nn.Linear `child` (attribute name `name`) by fn(child, name) unless it returns None"""
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            new = fn(child, name)
            if new is not None:
                setattr(module, name, new)
        else:
            wrap_linears(child, fn)

class AdapterBlock(nn.Module):
    """Houlsby-style bottleneck adapter after a frozen block: x + up(GELU(down(x)))"""
    def __init__(self, block, d_model, dim):
        super().__init__()
        self.block = block
        self.down, self.up = nn.Linear(d_model, dim), nn.Linear(dim, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = self.block(x)
        return x + self.up(F.gelu(self.down(x)))

# %% 6. CHUNK ENCODER + DOCUMENT POOLING + CLASSIFIER
def emb_module(bb):
    return bb.wte if hasattr(bb, "wte") else bb.token_embedding


class CJPEModel(nn.Module):
    def __init__(self, backbone, d_model, doc_pool):
        super().__init__()
        self.bb, self.doc_pool = backbone, doc_pool
        if doc_pool == "attn":
            self.att = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
        if doc_pool == "gru":
            self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.1)
        self.cls = nn.Linear(d_model, 2)

    def chunk_vectors(self, chunks, pool, autocast):
        vecs = []
        for b in range(0, len(chunks), WIN_BATCH):
            part = chunks[b:b + WIN_BATCH]
            T = max(map(len, part))
            x = torch.zeros(len(part), T, dtype=torch.long)
            for i, c in enumerate(part):
                x[i, :len(c)] = torch.tensor(c)          # right padding: safe for a causal model
            with autocast():
                h, _ = self.bb(x.to(DEVICE))
            for i, c in enumerate(part):
                vecs.append(h[i, len(c) - 1] if pool == "last" else h[i, :len(c)].mean(0))
        return torch.stack(vecs).float()                  # (n_chunks, d)

    def forward(self, chunks, pool, autocast):
        v = self.chunk_vectors(chunks, pool, autocast)
        if self.doc_pool == "attn":
            w = torch.softmax(self.att(v).squeeze(-1), 0)
            doc = (w.unsqueeze(-1) * v).sum(0)
        elif self.doc_pool == "gru":
            out, _ = self.gru(v.unsqueeze(0))
            doc = out[0].mean(0)
        else:
            doc = v.mean(0)
        return self.cls(self.drop(doc))                    # (2,)


def build_cjpe_model(model_type, p1_cfg, p1_state, method, doc_pool, keep_every=None):
    bb = BUILDERS[model_type](p1_cfg)
    if p1_state is not None:
        bb.load_state_dict(p1_state)
    bb.lm_head = nn.Identity()
    if keep_every:
        bb.blocks = nn.ModuleList(list(bb.blocks)[::keep_every])
    d = emb_module(bb).weight.shape[1]
    m = CJPEModel(bb, d, doc_pool)
    if method in ("full",):
        pass
    elif method == "partial":
        for p in bb.parameters(): p.requires_grad = False
        for blk in bb.blocks[FREEZE_BOTTOM:]:
            for p in blk.parameters(): p.requires_grad = True
        for n, p in bb.named_parameters():
            if not n.startswith("blocks.") and p.dim() == 1: p.requires_grad = True    # final norm
    elif method == "unfreeze":
        for p in bb.parameters(): p.requires_grad = False   # unfrozen top-down during training (Cell 8)
    elif method in ("lora", "lorma", "adalora"):
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora":
            wrap_linears(bb.blocks, lambda lin, n: LoRALinear(lin, LORA_R, LORA_ALPHA))
        elif method == "lorma":
            wrap_linears(bb.blocks, lambda lin, n: LoRMALinear(lin, LORA_R, LORA_ALPHA))
        else:
            wrap_linears(bb.blocks, lambda lin, n: AdaLoRALinear(lin, ADA_R_INIT, LORA_ALPHA))
    elif method == "adapter":
        for p in bb.parameters(): p.requires_grad = False
        bb.blocks = nn.ModuleList([AdapterBlock(b, d, ADAPTER_DIM) for b in bb.blocks])
    else:
        raise ValueError(method)
    return m

# %% 7. LOAD PHASE-1 WEIGHTS (+ KD teacher)
def as_ckpt_file(path):
    """Kaggle sometimes auto-extracts an uploaded .pt (it is a zip inside) into a FOLDER.
    If `path` is such a folder, re-pack it into a normal .pt file that torch.load can read."""
    if not os.path.isdir(path):
        return path
    root = next((r for r, _, f in os.walk(path) if "data.pkl" in f), None)
    assert root, f"{path} is a folder without data.pkl - not an extracted checkpoint"
    out = os.path.join("/kaggle/working", os.path.basename(path.rstrip("/")) + "_repacked.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, _, files in os.walk(root):
            for f in files:
                full = os.path.join(r, f)
                z.write(full, "archive/" + os.path.relpath(full, root))
    print(f"Re-packed extracted checkpoint folder {path} -> {out}")
    return out
ck = torch.load(as_ckpt_file(PHASE1_CKPT), map_location="cpu")
P1_CFG = ck["config"]
print("Phase-1 config:", P1_CFG)
assert P1_CFG["vocab_size"] == len(tokenizer), "Tokenizer doesn't match this Phase-1 model"
assert MAX_LEN <= P1_CFG.get("block_size", P1_CFG.get("max_seq_len")), "MAX_LEN > Phase-1 context"

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

RUN_CFG = dict(model_type=MODEL_TYPE, ft_method=FT_METHOD, doc_pool=DOC_POOL, chunk_pool=CHUNK_POOL,
               section=SECTION, n_chunks=N_CHUNKS, max_len=MAX_LEN, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
               ada_r_init=ADA_R_INIT, adapter_dim=ADAPTER_DIM, freeze_bottom=FREEZE_BOTTOM,
               keep_every=KD_KEEP_EVERY if KD_TEACHER else None, phase1_cfg=P1_CFG)

teacher = None
if KD_TEACHER:
    t = torch.load(as_ckpt_file(KD_TEACHER), map_location="cpu")
    tc = t["run_cfg"]
    assert (tc["section"], tc["n_chunks"], tc["max_len"]) == (SECTION, N_CHUNKS, MAX_LEN), \
        "teacher must read the same part of the judgment (SECTION / N_CHUNKS / MAX_LEN)"
    for k in ("lora_r", "lora_alpha", "ada_r_init", "adapter_dim", "freeze_bottom"):
        globals()[k.upper()] = tc[k]          # rebuild the teacher with ITS settings
    teacher = build_cjpe_model(tc["model_type"], tc["phase1_cfg"], None, tc["ft_method"], tc["doc_pool"],
                               tc.get("keep_every"))
    teacher.load_state_dict(t["state"])
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters(): p.requires_grad = False
    T_POOL = tc["chunk_pool"]
    n_teacher = sum(p.numel() for p in teacher.parameters())
    print(f"KD teacher: {tc['model_type']} + {tc['ft_method']} ({n_teacher:,} params)")
    FT_METHOD = RUN_CFG["ft_method"] = "full"      # student = smaller body, fully trained

model = build_cjpe_model(MODEL_TYPE, P1_CFG, ck["model"], FT_METHOD, DOC_POOL,
                         KD_KEEP_EVERY if KD_TEACHER else None).to(DEVICE)
del ck
n_total = sum(p.numel() for p in model.parameters())
def n_trainable():
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{FT_METHOD}{' (KD student)' if KD_TEACHER else ''}: trainable {n_trainable():,} / {n_total:,} "
      f"({100 * n_trainable() / n_total:.2f}%) | section={SECTION} {N_CHUNKS}x{MAX_LEN} | "
      f"chunk_pool={CHUNK_POOL} doc_pool={DOC_POOL}")
if KD_TEACHER:
    print(f"KD: student {n_total:,} params = {100 * n_total / n_teacher:.0f}% of teacher")

# %% 8. TRAINING
def macro_f1(y, p):
    out = {}
    for c in (0, 1):
        tp = sum(1 for a, b in zip(y, p) if a == c and b == c)
        fp = sum(1 for a, b in zip(y, p) if a != c and b == c)
        fn = sum(1 for a, b in zip(y, p) if a == c and b != c)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        out[c] = (pr, rc, 2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return {"f1": (out[0][2] + out[1][2]) / 2, "precision": (out[0][0] + out[1][0]) / 2,
            "recall": (out[0][1] + out[1][1]) / 2, "f1_class0": out[0][2], "f1_class1": out[1][2]}

counts = Counter(d["label"] for d in splits["train"])
class_w = (torch.tensor([len(splits["train"]) / (2 * max(counts[c], 1)) for c in (0, 1)], device=DEVICE)
           if CLASS_WEIGHT else None)
print("train class counts:", dict(counts), "| class weights:", None if class_w is None else class_w.tolist())

new_ids = {id(p) for n, p in model.named_parameters()
           if not n.startswith("bb.") or n.endswith((".A", ".B", ".P", ".Q", ".lam"))
           or ".down." in n or ".up." in n}
body = [p for p in model.parameters() if id(p) not in new_ids]
groups = [{"params": [p for p in model.parameters() if id(p) in new_ids and p.requires_grad], "lr": LR_NEW},
          {"params": body if FT_METHOD == "unfreeze" else [p for p in body if p.requires_grad], "lr": LR_BODY}]
optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
train_docs = splits["train"]
total_steps = EPOCHS * math.ceil(len(train_docs) / DOCS_PER_STEP)
warmup = max(1, int(0.06 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

# gradual unfreezing: block k (from the TOP) becomes trainable at step k * total / (L + 1); embeddings never
N_BLOCKS = len(model.bb.blocks)
def apply_unfreeze(step):
    k = min(N_BLOCKS, int(step / max(1, total_steps / (N_BLOCKS + 1))))
    for i, blk in enumerate(model.bb.blocks):
        for p in blk.parameters():
            p.requires_grad = i >= N_BLOCKS - k
    for n, p in model.bb.named_parameters():
        if not n.startswith("blocks.") and p.dim() == 1:
            p.requires_grad = k > 0          # final norm with the first unfrozen block
    return k

ada_mods = [m_ for m_ in model.modules() if isinstance(m_, AdaLoRALinear)]
ADA_BUDGET0 = sum(m_.lam.numel() for m_ in ada_mods)
ADA_TARGET = ADA_R_TARGET * len(ada_mods)
ADA_T0, ADA_T1, ADA_EVERY, ADA_BETA = int(0.1 * total_steps), int(0.7 * total_steps), 10, 0.85
def adalora_update(step):
    for m_ in ada_mods:
        if m_.lam.grad is not None:
            m_.importance.mul_(ADA_BETA).add_((1 - ADA_BETA) * (m_.lam * m_.lam.grad).abs().detach())
    if step < ADA_T0 or step % ADA_EVERY: return
    prog = min(1.0, (step - ADA_T0) / max(1, ADA_T1 - ADA_T0))
    budget = int(ADA_TARGET + (ADA_BUDGET0 - ADA_TARGET) * (1 - prog) ** 3)
    allimp = torch.cat([m_.importance for m_ in ada_mods])
    if budget < allimp.numel():
        thr = torch.topk(allimp, budget).values[-1]
        for m_ in ada_mods:
            m_.mask.copy_((m_.importance >= thr).float())

@torch.no_grad()
def predict_probs(m, docs, pool=CHUNK_POOL):
    m.eval()
    return [torch.softmax(m(d["chunks"], pool, autocast).float(), -1)[1].item() for d in docs]

def best_threshold(y, probs):
    if not TUNE_THRESHOLD:
        return 0.5
    cands = sorted(set(round(p, 3) for p in probs)) or [0.5]
    cands = cands[:: max(1, len(cands) // 200)] + [0.5]
    # best Macro-F1; among ties, the threshold closest to 0.5 (less over-fitted to dev)
    return max(cands, key=lambda t: (round(macro_f1(y, [int(p >= t) for p in probs])["f1"], 6), -abs(t - 0.5)))

BEST_PATH = os.path.join(OUTPUT_DIR, "best_cjpe.pt")
EVAL_EVERY = max(1, math.ceil(len(train_docs) / DOCS_PER_STEP / 2))   # evaluate twice per epoch
best_f1, history, step, t0 = -1.0, [], 0, time.time()
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
if FT_METHOD == "unfreeze": print("unfrozen blocks:", apply_unfreeze(0))

DEV_SUB = random.Random(SEED + 1).sample(splits["dev"], min(EVAL_DEV_DOCS or len(splits["dev"]), len(splits["dev"])))

def evaluate_and_maybe_save(tag):
    global best_f1, best_res, BEST_T
    y = [d["label"] for d in DEV_SUB]
    probs = predict_probs(model, DEV_SUB)
    thr = best_threshold(y, probs)
    res = macro_f1(y, [int(p >= thr) for p in probs]); res["threshold"] = thr
    res["f1_at_0.5"] = macro_f1(y, [int(p >= 0.5) for p in probs])["f1"]
    history.append({"at": tag, **res})
    print(f"  [{tag}] dev Macro-F1 {res['f1']:.4f} (threshold {thr:.3f}; at 0.5: {res['f1_at_0.5']:.4f}) | "
          f"F1 class0 {res['f1_class0']:.3f} class1 {res['f1_class1']:.3f} | {time.time() - t0:.0f}s")
    if res["f1"] > best_f1:
        best_f1, best_res, BEST_T = res["f1"], res, thr
        torch.save({"state": model.state_dict(), "run_cfg": RUN_CFG, "threshold": thr}, BEST_PATH)
        print("  >> best model saved")
    model.train()

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_docs)
    tot, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for i, d in enumerate(train_docs):
        logits = model(d["chunks"], CHUNK_POOL, autocast).float()
        y = torch.tensor(d["label"], device=DEVICE)
        loss = F.cross_entropy(logits.unsqueeze(0), y.unsqueeze(0), weight=class_w)
        if teacher is not None:
            with torch.no_grad():
                t_logits = teacher(d["chunks"], T_POOL, autocast).float()
            kl = F.kl_div(F.log_softmax(logits / KD_T, -1), F.softmax(t_logits / KD_T, -1), reduction="sum")
            loss = KD_ALPHA * loss + (1 - KD_ALPHA) * KD_T ** 2 * kl
        if ada_mods:
            loss = loss + 0.1 * sum(m_.orth_reg() for m_ in ada_mods) / len(ada_mods)
        scaler.scale(loss / DOCS_PER_STEP).backward()
        tot += loss.item(); n += 1
        if (i + 1) % DOCS_PER_STEP == 0 or i + 1 == len(train_docs):
            scaler.unscale_(optimizer)
            if ada_mods: adalora_update(step)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], 1.0)
            scaler.step(optimizer); scaler.update(); sched.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if FT_METHOD == "unfreeze":
                k = apply_unfreeze(step)
            if step % 50 == 0:
                rate = (time.time() - t0) / step
                print(f"epoch {epoch} step {step}/{total_steps} | loss {tot / n:.4f} | "
                      f"ETA training {rate * (total_steps - step) / 60:.0f} min"
                      + (f" | unfrozen blocks {k}" if FT_METHOD == "unfreeze" else ""))
            if step % EVAL_EVERY == 0:
                evaluate_and_maybe_save(f"epoch {epoch} step {step}")

TRAIN_MIN = (time.time() - t0) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else None
print(f"\nBest dev-subset Macro-F1 {best_f1:.4f} | {TRAIN_MIN:.1f} min | peak GPU {PEAK_GB} GB")

# re-score the best checkpoint on the FULL dev set and set the final threshold there
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state"])
y_full = [d["label"] for d in splits["dev"]]
p_full = predict_probs(model, splits["dev"])
BEST_T = best_threshold(y_full, p_full)
best_res = macro_f1(y_full, [int(p >= BEST_T) for p in p_full]); best_res["threshold"] = BEST_T
best_res["f1_at_0.5"] = macro_f1(y_full, [int(p >= 0.5) for p in p_full])["f1"]
torch.save({**ckpt, "threshold": BEST_T}, BEST_PATH)
print(f"FULL dev ({len(y_full)} cases): Macro-F1 {best_res['f1']:.4f} at threshold {BEST_T:.3f} "
      f"(at 0.5: {best_res['f1_at_0.5']:.4f}) | F1 class0 {best_res['f1_class0']:.3f} class1 {best_res['f1_class1']:.3f}")

# %% 9. TEST PREDICTIONS + prediction.json + ZIP
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE)["state"])
t1 = time.time()
probs = predict_probs(model, splits["test"])
INFER_SEC_PER_CASE = (time.time() - t1) / len(splits["test"])
preds = [{"id": d["id"], "pred": int(p >= BEST_T)} for d, p in zip(splits["test"], probs)]
assert len(preds) == len(splits["test"]) and all(p["pred"] in (0, 1) for p in preds)
print(f"{len(preds)} test cases | predicted classes {dict(Counter(p['pred'] for p in preds))} | "
      f"{INFER_SEC_PER_CASE * 1000:.0f} ms per case")

pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f)
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Submission ready:", zip_path)

# %% 10. RUN SUMMARY (experiment log / report)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({**{k: v for k, v in RUN_CFG.items() if k != "phase1_cfg"}, "kd_teacher": KD_TEACHER,
               "trainable_params_end": n_trainable(), "total_params": n_total,
               "teacher_params": n_teacher if KD_TEACHER else None,
               "train_docs": len(train_docs), "epochs": EPOCHS, "lr_body": LR_BODY, "lr_new": LR_NEW,
               "class_weight": CLASS_WEIGHT, "dev_best": best_res, "history": history,
               "train_minutes": round(TRAIN_MIN, 1), "peak_gpu_gb": PEAK_GB,
               "inference_ms_per_case": round(INFER_SEC_PER_CASE * 1000, 1)}, f, indent=2)
print("Done.")

Device: cuda Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/tmp/cs781_data/Phase_4_CJPE/Train/dev.json 80.3 MB
/tmp/cs781_data/Phase_4_CJPE/Train/train.json 736.6 MB
/tmp/cs781_data/Phase_4_CJPE/Test/test.json 35.3 MB

Train/train.json: list with 37387 entries | first: {"id": "2020_1", "text": " Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters and thereby affirming the decision of the Single Judge of the High Court ...

Train/dev.json: list with 3505 entries | first: {"id": "1989_75", "text": "civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.10.1987 of t\nhe\ncentral administrative tribunal chandigarh in appln. t.\n--\n1055 of 1986.\nwith\n slp  civil  number 6998 of 1988 \n\nc. mahajan mrs. indu goswami c.v. ...

Test/test.json: list with 1517 entries | first: {"id": "1986_397", "text": "civil appellate jurisdiction civil appeal number 1257\n